# SNOWFLAKE.CORTEX.COMPLETE — Prompt Engineering Patterns

## Use Case Overview

`SNOWFLAKE.CORTEX.COMPLETE(model, prompt)` is the foundational GenAI building block in Snowflake. Every other AI function (`AI_CLASSIFY`, `AI_SENTIMENT`, etc.) is built on top of it. Understanding how to use `CORTEX.COMPLETE` directly gives you unlimited flexibility for custom use cases.

**SA use case:** Demonstrate the full power of LLMs in SQL — from simple text generation to structured JSON output to few-shot classification — all without leaving Snowflake.

**What we'll cover:**
1. **Pattern 1 — Simple instruction**: generate content from a prompt
2. **Pattern 2 — Data-grounded generation**: pass column values into the prompt as context
3. **Pattern 3 — Structured JSON output**: force the LLM to return valid JSON, then `PARSE_JSON` it
4. **Pattern 4 — Few-shot classification**: provide examples in the prompt for better accuracy
5. **Pattern 5 — Model comparison**: run the same prompt across different models

In [ ]:
import os
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "SFCOGSOPS-SNOWHOUSE_AWS_US_WEST_2"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Pattern 1 — Simple Instruction Prompt

The most basic usage: pass a text prompt, get a text response.
`SNOWFLAKE.CORTEX.COMPLETE(model_name, prompt_string)` returns a VARCHAR.

In [ ]:
result = pd.read_sql("""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-haiku',
        'Write a one-sentence elevator pitch for Snowflake Cortex AI targeted at a CFO.'
    ) AS response
""", conn)
print(result["RESPONSE"][0])

## Pattern 2 — Data-Grounded Generation

Pass column values from your table into the prompt using string concatenation.
This is how you generate personalised content, summaries, or responses at scale — one per row.

In [ ]:
grounded_df = pd.read_sql("""
    SELECT
        review_id,
        product_name,
        star_rating,
        review_text,
        SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-haiku',
            'You are a customer success manager. Write a professional, empathetic response to this customer review in 2 sentences max.\n\nReview (' || star_rating::STRING || ' stars): ' || review_text
        ) AS suggested_response
    FROM product_reviews
    WHERE star_rating <= 2
    ORDER BY review_id
    LIMIT 5
""", conn)
for _, row in grounded_df.iterrows():
    print(f"\n--- Review #{row['REVIEW_ID']} ({row['STAR_RATING']} stars) ---")
    print(f"Customer: {row['REVIEW_TEXT']}")
    print(f"Suggested Response: {row['SUGGESTED_RESPONSE']}")

## Pattern 3 — Structured JSON Output

Force the LLM to return a JSON object by including a schema in the prompt.
Then use `PARSE_JSON()` to turn the response into a VARIANT for downstream processing.

This is the foundation for building structured enrichment pipelines.

In [ ]:
json_output_df = pd.read_sql("""
    WITH raw AS (
        SELECT
            review_id,
            product_name,
            review_text,
            SNOWFLAKE.CORTEX.COMPLETE(
                'claude-3-5-haiku',
                'Analyse this product review and return ONLY a valid JSON object with these exact keys: "main_issue" (string), "tone" (positive/neutral/negative), "actionable" (true/false), "keywords" (array of up to 3 strings).\n\nReview: ' || review_text
            ) AS raw_json
        FROM product_reviews
        LIMIT 5
    )
    SELECT
        review_id,
        product_name,
        TRY_PARSE_JSON(raw_json) AS structured,
        TRY_PARSE_JSON(raw_json):main_issue::STRING   AS main_issue,
        TRY_PARSE_JSON(raw_json):tone::STRING         AS tone,
        TRY_PARSE_JSON(raw_json):actionable::BOOLEAN  AS actionable,
        TRY_PARSE_JSON(raw_json):keywords             AS keywords
    FROM raw
""", conn)
json_output_df

## Pattern 4 — Few-Shot Classification

Provide examples in the prompt to dramatically improve classification accuracy for domain-specific labels that the model hasn't been trained on specifically.

In [ ]:
few_shot_prompt = """Classify the following order comment into exactly one category. 
Categories: FULFILLMENT_ISSUE, PAYMENT_DISPUTE, DELIVERY_DELAY, PRODUCT_QUALITY, NO_ISSUE

Examples:
Comment: "the package was damaged on arrival" -> PRODUCT_QUALITY
Comment: "charged twice on my card" -> PAYMENT_DISPUTE  
Comment: "shipped three weeks late" -> DELIVERY_DELAY
Comment: "item was missing from the box" -> FULFILLMENT_ISSUE
Comment: "everything arrived perfectly" -> NO_ISSUE

Comment: "{comment}" -> """

few_shot_df = pd.read_sql(f"""
    SELECT
        order_key,
        order_comment,
        TRIM(SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-haiku',
            REPLACE('{few_shot_prompt}', '{{comment}}', order_comment)
        )) AS category
    FROM order_comments_vw
    LIMIT 10
""", conn)
few_shot_df

## Pattern 5 — Model Comparison

Snowflake supports multiple LLM models. Run the same prompt across models to compare quality, speed, and token usage.

Available models include: `claude-3-5-haiku`, `claude-3-5-sonnet`, `mistral-large2`, `llama3.1-70b`, `llama3.1-405b`, `snowflake-arctic`

In [ ]:
comparison_prompt = "In exactly 2 sentences, explain what a Snowflake Semantic View is to a business analyst."

comparison_df = pd.read_sql(f"""
    SELECT model_name, response FROM (
        SELECT 'claude-3-5-haiku' AS model_name,
               SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-haiku', '{comparison_prompt}') AS response
        UNION ALL
        SELECT 'mistral-large2',
               SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', '{comparison_prompt}')
        UNION ALL
        SELECT 'llama3.1-70b',
               SNOWFLAKE.CORTEX.COMPLETE('llama3.1-70b', '{comparison_prompt}')
    )
""", conn)

for _, row in comparison_df.iterrows():
    print(f"\n=== {row['MODEL_NAME']} ===")
    print(row["RESPONSE"])

## Summary — Choosing the Right Approach

| Need | Use |
|---|---|
| Simple generation or classification | `CORTEX.COMPLETE` with clear instruction |
| Per-row content generation | `CORTEX.COMPLETE` with column values in prompt |
| Structured output for pipelines | `CORTEX.COMPLETE` + `TRY_PARSE_JSON` |
| Domain-specific categories | Few-shot examples in prompt |
| Pre-built intent (sentiment, classify, etc.) | Dedicated `AI_*` functions (simpler, optimised) |
| Best-model selection | Compare with `CORTEX.COMPLETE` across models |